# 02 · Calibration and training

1. Fit the per-device **lens-position → distance** calibration on the training captures.
2. Train every model listed in `RUNS`. The default is three:
   - `unet_multi`: the U-Net with the focus input,
   - `image_only`: the same U-Net with no focus input (the ablation),
   - `unet_embed`: the U-Net with the focus input and frozen **DINOv2-small** features.

Only sessions whose manifest says `split: train` are used. The test zip is never opened here.
No model ever receives depth as an input. The first `unet_embed` run downloads the DINOv2
weights (about 90 MB).

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "python"))  # use the package without installing it
DATA = Path("../data")      # session zips exported from the phone
MODELS = Path("../models")  # checkpoints, calibration.json, UNetUltra.mlpackage
MODELS.mkdir(exist_ok=True)

In [ ]:
# checkpoint name -> architecture (any name registered in unetultra.models.ARCHITECTURES)
RUNS = {
    "model": "unet_multi",
    "model_image_only": "image_only",
    "model_embed": "unet_embed",
}
EPOCHS = 60
BATCH_SIZE = 8
DEPTH_WEIGHT = 0.5          # weight of the depth loss against the mask loss
OBJECT_DEPTH_WEIGHT = 1.0   # weight of the depth error averaged over object pixels alone (0 = plain L1 over all valid pixels)
VAL_SESSIONS = None         # e.g. ["../data/session_....zip"]; None holds out about a quarter of the sessions

In [ ]:
from unetultra.session import find_sessions
from unetultra.train import split_sessions

all_train_paths = find_sessions(DATA, "train")
SINGLE_SESSION = len(all_train_paths) == 1 and not VAL_SESSIONS
if SINGLE_SESSION:
    train_paths = val_paths = all_train_paths
    print("One training session:", all_train_paths[0])
    print("Its last 20% of samples are held out for validation. They come from the same room and day,")
    print("so validation numbers will flatter the model; trust only the test session in 03_evaluate.")
else:
    train_paths, val_paths = split_sessions(all_train_paths, VAL_SESSIONS)
    print("train:", *train_paths, sep="\n  ")
    print("validation:", *val_paths, sep="\n  ")

## Lens calibration

Each point is one capture: the lens reading after focus settled, against the LiDAR depth at the
tap (median of valid pixels in a small window). The fit is monotonic (isotonic regression on
1/distance). Residuals should scatter around zero with no trend.

In [ ]:
from unetultra import plots
from unetultra.calibration import fit_from_sessions

calibration, points = fit_from_sessions(train_paths)
calibration.save(MODELS / "calibration.json")
print(f"{calibration.n_samples} samples, RMSE {calibration.rmse_m * 100:.2f} cm, device {calibration.device_model}")
plots.calibration_fit(calibration, points);

## Datasets

Accepted samples only. Pre-processing takes the centre square of each frame and resizes it to
512×512, with the intrinsics transformed to match. Samples whose object was tapped outside that
square cannot be used and are listed as `excluded_outside_crop`.

In [ ]:
from torch.utils.data import Subset
from unetultra.dataset import SessionDataset
from unetultra.models import uses_meta
from unetultra.train import split_single_session


def datasets(architecture):
    """Train and validation sets for one architecture (an image-only model gets a zeroed meta input)."""
    use_meta = uses_meta(architecture)
    train = SessionDataset(train_paths, calibration, augment=True, use_meta=use_meta)
    val = SessionDataset(val_paths, calibration, use_meta=use_meta)
    if SINGLE_SESSION:
        train_indices, val_indices = split_single_session(train)
        return Subset(train, train_indices), Subset(val, val_indices), train
    return train, val, train


train_set, val_set, full_train = datasets("unet_multi")
background = sum(int(train_set[i]["instances"].max()) == 0 for i in range(len(train_set)))
print(f"train {len(train_set)} samples ({full_train.skipped} skipped as not accepted), validation {len(val_set)}")
for path, sample_id, reason in full_train.excluded:
    print(f"  {reason}: {Path(path).name} sample {sample_id}")
print(f"background-only share of training samples: {background / max(len(train_set), 1):.0%} (aim for about 10%)")
print("classes:", full_train.classes)

## Train

One line per epoch; `*` marks a new best checkpoint (lowest validation object-distance error).

In [ ]:
from unetultra.train import TrainConfig, fit

histories = {}
for name, architecture in RUNS.items():
    config = TrainConfig(architecture=architecture, epochs=EPOCHS, batch_size=BATCH_SIZE,
                         depth_weight=DEPTH_WEIGHT, object_depth_weight=OBJECT_DEPTH_WEIGHT)
    run_train, run_val, _ = datasets(architecture)
    histories[name], checkpoint = fit(run_train, run_val, config, MODELS, name=name)
    print("best checkpoint:", checkpoint, "\n")

## Learning curves

In [ ]:
plots.learning_curves(histories);